# Save Qwen3-8B to Kaggle Dataset — Run Once

Downloads Qwen3-8B weights from HuggingFace and pushes them as a Kaggle dataset.
After this runs once, all 3 agent kernels load from `/kaggle/input/qwen3-8b-cache/`
instead of re-downloading every run.

**Time:** ~15 min (download) + ~5 min (upload to Kaggle)
**Run once per account — never again.**

**After running:**
1. Go to kaggle.com/datasets/builder117/qwen3-8b-cache
2. Add it as a Data Source to each agent kernel
3. Future agent runs load in ~2 min instead of ~15 min

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "huggingface_hub>=0.23.0", "kaggle"], check=True)
print('Install complete.')

In [ ]:
import os, json
from pathlib import Path
from huggingface_hub import snapshot_download

HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('KAGGLE_SECRET_HF_TOKEN', '')
KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', 'builder117')
MODEL_ID = 'Qwen/Qwen3-8B'
SAVE_DIR = Path('/kaggle/working/qwen3-8b-cache')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Downloading {MODEL_ID} to {SAVE_DIR} ...')
print('This takes ~15 min on Kaggle T4. Watch disk usage in the right panel.')

snapshot_download(
    repo_id=MODEL_ID,
    local_dir=str(SAVE_DIR),
    token=HF_TOKEN if HF_TOKEN else None,
    ignore_patterns=['*.msgpack', '*.h5', 'flax_*', 'tf_*'],  # skip non-PyTorch weights
)

files = list(SAVE_DIR.rglob('*'))
total_gb = sum(f.stat().st_size for f in files if f.is_file()) / 1e9
print(f'Downloaded {len(files)} files, {total_gb:.1f} GB')

In [ ]:
# Write Kaggle dataset metadata
meta = {
    'title': 'Qwen3-8B Model Cache',
    'id': f'{KAGGLE_USERNAME}/qwen3-8b-cache',
    'licenses': [{'name': 'apache-2.0'}],
    'description': 'Qwen3-8B model weights cached for fast load in agent kernels. Load with BitsAndBytesConfig nf4 for INT4 quantization.'
}
(SAVE_DIR / 'dataset-metadata.json').write_text(json.dumps(meta, indent=2))
print('Metadata written.')
print(f'Dataset will be: kaggle.com/datasets/{KAGGLE_USERNAME}/qwen3-8b-cache')

In [ ]:
import subprocess

# Kaggle CLI uses KAGGLE_USERNAME + KAGGLE_KEY env vars (set as Kaggle secrets)
print('Pushing dataset to Kaggle...')
result = subprocess.run(
    ['kaggle', 'datasets', 'create', '-p', str(SAVE_DIR), '--dir-mode', 'zip'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    # Dataset may already exist — try update instead
    result2 = subprocess.run(
        ['kaggle', 'datasets', 'version', '-p', str(SAVE_DIR), '-m', 'model weights'],
        capture_output=True, text=True
    )
    print(result2.stdout)
else:
    print(f'Done! Dataset at: kaggle.com/datasets/{KAGGLE_USERNAME}/qwen3-8b-cache')
    print()
    print('Next steps:')
    print('  1. Open each agent kernel (red_team, blue_team, orchestrator)')
    print(f'  2. Data → Add Data → {KAGGLE_USERNAME}/qwen3-8b-cache')
    print('  3. Model will load from /kaggle/input/qwen3-8b-cache/ in ~2 min')